# GameTheory-04j — Rationalisabilité : ce que la seule rationalité peut délier

**Version jumelle** : [Python](GameTheory-04j-Rationalisabilite-Python.ipynb)

**Navigation** : [<< 04i-Utilite-Esperee-Risque](GameTheory-04i-Utilite-Esperee-Risque-Python.ipynb) | [Index](README.md) | [04k-Equilibre-Correle >>](GameTheory-04k-Equilibre-Correle-Python.ipynb)

Le carnet [GameTheory-04-NashEquilibrium-Python.ipynb](GameTheory-04-NashEquilibrium-Python.ipynb) calcule des équilibres de Nash en stratégies mixtes. L'équilibre exige une **cohérence mutuelle** : la stratégie de chaque joueur est une meilleure réponse à celle de l'autre, et réciproquement. Que reste-t-il si on abandonne cette exigence et qu'on ne garde que la **rationalité** — chaque joueur joue une meilleure réponse à *une* croyance sur son adversaire — itérée par connaissance commune ? La réponse est l'ensemble des **stratégies rationalisables** : ce carnet le construit, le calcule, et mesure l'écart exact qui le sépare des équilibres du palier 04.

Trois idées structurent le parcours :

1. une stratégie peut être **condamnée sans être dominée par une stratégie pure** — la Table 5.8 de Bonanno livre un exemple qu'on vérifie au calcul ;
2. le **théorème de Pearce (1984)** — jamais-meilleure-réponse *si et seulement si* dominée par une stratégie mixte — est vérifié numériquement **dans les deux sens**, par programmation linéaire ;
3. l'**IDSDS cardinal** (itération de la suppression) définit les stratégies rationalisables ; sur les jeux mêmes du palier 04 (Dilemme du Prisonnier, Battle of the Sexes, chasse au cerf, Matching Pennies), on mesure que l'inclusion dans les équilibres est **stricte sur les trois jeux étudiés où aucune dominance n'amorce la purge**.

**Position dans la série** : accrétion du palier 04 — la rationalisabilité est la question « que délie la seule rationalité, sans équilibre ? » que le calcul des mixtes suppose tranchée. Le carnet 04i (Utilité Espérée et Risque) construit les utilités cardinales que la dominance par stratégies mixtes requiert ; [DecInfer-09-Theories-Decision-Predicteur-Python.ipynb](../Probas/DecisionTheory/DecInfer/DecInfer-09-Theories-Decision-Predicteur-Python.ipynb) lit les mixtes comme des croyances ; [GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb](GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb) construit les opérateurs de connaissance que la lecture épistémique (section 5) consomme. Référence bibliographique du carnet : Bonanno (2015), ch. 5.4.
**Source primaire** : carnet Python versionné ; bibliographie héritée sans nouvelle lecture revendiquée. Prérequis : Python.NET 3.1.0, CPython compatible, OR-Tools GLOP et Nashpy ; le [chargeur partagé](../Config/PythonNetLoader.cs) sélectionne CPython.
## Prérequis
- [04-NashEquilibrium](GameTheory-04-NashEquilibrium-Python.ipynb) — les équilibres de Nash en stratégies mixtes


## Références

Bibliographie héritée du carnet Python versionné ; les calculs LP C# utilisent OR-Tools GLOP et les équilibres Nashpy.


In [1]:
#r "nuget: pythonnet,3.1.0"
#r "nuget: Google.OrTools,9.15.6755"
#load "../Config/PythonNetLoader.cs"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using Python.Runtime;
using Google.OrTools.LinearSolver;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
// Le chargeur commun initialise un CPython qui importe les moteurs scientifiques.
PythonNetLoader.Initialize(typeof(PythonEngine).Assembly,"nashpy");
using(Py.GIL())
{
    using var np=Py.Import("numpy");using var nash=Py.Import("nashpy");
    Console.WriteLine($"NumPy {np.GetAttr("__version__")} | Nashpy {nash.GetAttr("__version__")}");
}


Installed Packages Google.OrTools, 9.15.6755 pythonnet, 3.1.0

NumPy 2.5.4 | Nashpy 0.0.43


## 1. Dominée par une stratégie mixte : ce que l'IDSDS ordinal ne voit pas

Le chapitre 1 du manuel définit l'**IDSDS ordinal** : supprimer itérativement les stratégies strictement dominées **par une autre stratégie pure**. Avec des paiements cardinaux (construits au carnet 04i), une stratégie peut être condamnée — jamais jouée par un joueur rationnel — sans être dominée par aucune stratégie pure. La Table 5.8 de Bonanno en est le témoin, avec les paiements du Joueur 1 en premier :

| | D | E |
|---|---|---|
| **A** | 0 , 1 | 4 , 0 |
| **B** | 1 , 2 | 1 , 4 |
| **C** | 2 , 0 | 0 , 1 |

In [2]:
// Table 5.8 : B n'est dominée par aucune ligne pure, mais par une mixture.
var t58 = new double[,]
{
    { 0, 4 },
    { 1, 1 },
    { 2, 0 },
};
var rowNames = new[] { "A", "B", "C" };
var colNames = new[] { "D", "E" };
foreach (int r in new[] { 0, 2 })
{
    var diff = Enumerable.Range(0, 2).Select(j => t58[r, j] - t58[1, j]).ToArray();
    Console.WriteLine(
        $"{rowNames[r]} - B : [{string.Join(", ", diff)}] ; domination stricte : {diff.All(x => x > 0)}"
    );
}
var weights = new[] { 1.0 / 3, 0, 2.0 / 3 };
var gains = Enumerable
    .Range(0, 2)
    .Select(j => Enumerable.Range(0, 3).Sum(i => weights[i] * t58[i, j]))
    .ToArray();
Console.WriteLine("Gain de B contre q=1/3 : 1");
Console.WriteLine(
    $"Mixte (1/3,0,2/3) : [{gains[0]:F6}, {gains[1]:F6}] ; marge minimale = {gains.Min() - 1:F6}"
);


A - B : [-1, 3] ; domination stricte : False


C - B : [1, -1] ; domination stricte : False


Gain de B contre q=1/3 : 1


Mixte (1/3,0,2/3) : [1.333333, 1.333333] ; marge minimale = 0.333333


### Interprétation : condamnée sans être dominée à l'œil nu

La lecture des sorties dit exactement le phénomène : ni A ni C ne domine B (le premier bloc rend un `False` sur les deux colonnes), le gain de B est **constant égal à 1** quelle que soit la croyance de l'adversaire, et la mixte $\frac{1}{3}A + \frac{2}{3}C$ rapporte $\frac{4}{3}$ contre **les deux** colonnes — une marge de $\frac{1}{3}$ sur B, partout. Autrement dit : quel que soit ce que le Joueur 1 croit que le Joueur 2 fera, jouer B est strictement irrationnel, alors qu'aucune stratégie pure ne la domine. C'est l'information **cardinale** (les valeurs 0, 1, 2, 4, pas seulement leur ordre) qui rend la condamnation visible : c'est le pont direct vers le carnet 04i sur les utilités espérées.

## 2. Le théorème de Pearce (1984), vérifié dans les deux sens

**Théorème 5.3 [Pearce, 1984]** (énoncé dans Bonanno, p. 201) : dans un jeu à deux joueurs en forme stratégique avec paiements cardinaux, une stratégie pure $s_i$ du Joueur $i$ n'est la meilleure réponse à **aucune** stratégie mixte de l'adversaire **si et seulement si** $s_i$ est strictement dominée par une stratégie mixte du Joueur $i$.

Les deux membres sont des problèmes de programmation linéaire, qu'on implémente indépendamment :

- **jamais-meilleure-réponse** : existe-t-il une croyance $q$ (distribution sur les colonnes) telle que $u(s, q) \geq u(s', q)$ pour toute autre ligne $s'$ ? On maximise la marge $t$ du système $\sum_j q_j\,(u(s,j) - u(s',j)) \geq t$ ; si le maximum est $< 0$, aucune croyance ne fait de $s$ une meilleure réponse.
- **dominée par une mixte** : existe-t-il une mixte $p$ sur les lignes telle que $\sum_i p_i\,u(i,j) \geq u(s,j) + t$ pour toute colonne $j$, avec $t > 0$ ?

In [3]:
// Les deux modèles LP sont construits et résolus nativement par GLOP.
PyList Vector(double[] v)
{
    var list = new PyList();
    foreach (double x in v)
    {
        using var value = new PyFloat(x);
        list.Append(value);
    }
    return list;
}
PyList Matrix(double[,] m)
{
    var rows = new PyList();
    for (int i = 0; i < m.GetLength(0); i++)
    {
        var row = Enumerable.Range(0, m.GetLength(1)).Select(j => m[i, j]).ToArray();
        using var pyrow = Vector(row);
        rows.Append(pyrow);
    }
    return rows;
}
// GLOP résout le LP natif : probabilités positives, dernière variable libre.
double[] SolveLinearProgram(double[] c, double[,] aub, double[] bub, double[,] aeq, double[] beq)
{
    using var solver = Solver.CreateSolver("GLOP") ?? throw new Exception("GLOP indisponible");
    int n = c.Length;
    var variables = new Variable[n];
    for (int i = 0; i < n; i++)
        variables[i] = solver.MakeNumVar(i == n - 1 ? double.NegativeInfinity : 0.0, double.PositiveInfinity, $"x{i}");
    for (int row = 0; row < bub.Length; row++)
    {
        var constraint = solver.MakeConstraint(double.NegativeInfinity, bub[row], $"inegalite{row}");
        for (int col = 0; col < n; col++)
            constraint.SetCoefficient(variables[col], aub[row, col]);
    }
    for (int row = 0; row < beq.Length; row++)
    {
        var constraint = solver.MakeConstraint(beq[row], beq[row], $"egalite{row}");
        for (int col = 0; col < n; col++)
            constraint.SetCoefficient(variables[col], aeq[row, col]);
    }
    var objective = solver.Objective();
    for (int i = 0; i < n; i++)
        objective.SetCoefficient(variables[i], c[i]);
    objective.SetMinimization();
    var status = solver.Solve();
    if (status != Solver.ResultStatus.OPTIMAL)
        throw new InvalidOperationException($"GLOP : statut {status}");
    var x = variables.Select(v => v.SolutionValue()).ToArray();
    if (x.Any(v => !double.IsFinite(v)) || x.Take(n - 1).Any(v => v < -1e-8))
        throw new InvalidOperationException("Solution GLOP non finie ou hors bornes");
    for (int row = 0; row < bub.Length; row++)
        if (Enumerable.Range(0,n).Sum(col => aub[row,col] * x[col]) > bub[row] + 1e-8)
            throw new InvalidOperationException("Contrainte GLOP violée");
    for (int row = 0; row < beq.Length; row++)
        if (Math.Abs(Enumerable.Range(0,n).Sum(col => aeq[row,col] * x[col]) - beq[row]) > 1e-8)
            throw new InvalidOperationException("Égalité GLOP violée");
    return x;
}
(double Margin, double[] Belief) BestResponseMargin(double[,] m, int s)
{
    int n = m.GetLength(0),
        k = m.GetLength(1);
    if (n < 2)
        throw new ArgumentException("Deux actions au minimum");
    var c = new double[k + 1];
    c[k] = -1;
    var a = new double[n - 1, k + 1];
    var b = new double[n - 1];
    int row = 0;
    for (int i = 0; i < n; i++)
        if (i != s)
        {
            for (int j = 0; j < k; j++)
                a[row, j] = m[i, j] - m[s, j];
            a[row, k] = 1;
            row++;
        }
    var eq = new double[1, k + 1];
    for (int j = 0; j < k; j++)
        eq[0, j] = 1;
    var x = SolveLinearProgram(c, a, b, eq, new[] { 1.0 });
    return (x[k], x.Take(k).ToArray());
}
(double Margin, double[] Mix) DominanceMargin(double[,] m, int s)
{
    int n = m.GetLength(0),
        k = m.GetLength(1);
    var c = new double[n + 1];
    c[n] = -1;
    var a = new double[k, n + 1];
    var b = new double[k];
    for (int j = 0; j < k; j++)
    {
        for (int i = 0; i < n; i++)
            a[j, i] = -m[i, j];
        a[j, n] = 1;
        b[j] = -m[s, j];
    }
    var eq = new double[1, n + 1];
    for (int i = 0; i < n; i++)
        eq[0, i] = 1;
    var x = SolveLinearProgram(c, a, b, eq, new[] { 1.0 });
    return (x[n], x.Take(n).ToArray());
}
Console.WriteLine("Table 5.8 : comparaison des deux LP GLOP");
for (int i = 0; i < 3; i++)
{
    var br = BestResponseMargin(t58, i);
    var dom = DominanceMargin(t58, i);
    Console.WriteLine(
        $"{rowNames[i]} : marge MR={br.Margin:F6}, marge dominée={dom.Margin:F6}, accord={(br.Margin < -1e-9) == (dom.Margin > 1e-9)}"
    );
}
var witness = DominanceMargin(t58, 1);
Console.WriteLine(
    $"Mixte dominatrice de B : [{string.Join(", ", witness.Mix.Select(x => x.ToString("F6")))}]"
);


Table 5.8 : comparaison des deux LP GLOP


A : marge MR=3.000000, marge dominée=0.000000, accord=True


B : marge MR=-0.333333, marge dominée=0.333333, accord=True


C : marge MR=1.000000, marge dominée=0.000000, accord=True


Mixte dominatrice de B : [0.333333, 0.000000, 0.666667]


### Interprétation : deux questions différentes, la même réponse

Les trois lignes d'accord `True` disent le théorème : pour **chaque** ligne de la Table 5.8, « aucune croyance n'en fait une meilleure réponse » (marge optimale négative) coïncide exactement avec « une stratégie mixte la domine strictement » (marge optimale positive) — et le solveur **retrouve de lui-même** la mixte $(\frac{1}{3}, 0, \frac{2}{3})$ du manuel. La coïncidence n'est pas un hasard de cet exemple : c'est l'équivalence de Pearce. Le lien conceptuel : chez le Joueur 1, une *croyance* sur l'adversaire et une *stratégie mixte* de l'adversaire sont le même objet mathématique (une distribution) — c'est spécifique aux deux joueurs, et l'exercice 3 (du manuel 5.14) montrera qu'à trois joueurs une croyance peut être **corrélée**, ce qu'un profil mixte indépendant ne peut pas être.

## 3. IDSDS cardinal et stratégies rationalisables

**Définition 5.6 (Cardinal IDSDS)** : supprimer du jeu, pour chaque joueur, les stratégies strictement dominées **par une stratégie mixte** de ce joueur ; réitérer sur le jeu réduit ; le survivant $G^\infty$ porte les **stratégies rationalisables**. La Figure 5.9 du manuel (p. 203) déroule la procédure sur un jeu $3 \times 3$ — on la rejoue avec le solveur LP, suppression par suppression :

| | D | E | F |
|---|---|---|---|
| **A** | 3 , 4 | 2 , 1 | 1 , 2 |
| **B** | 0 , 0 | 1 , 3 | 4 , 1 |
| **C** | 1 , 4 | 1 , 4 | 2 , 6 |

In [4]:
var g0a = new double[,]
{
    { 3, 2, 1 },
    { 0, 1, 4 },
    { 1, 1, 2 },
};
var g0b = new double[,]
{
    { 4, 1, 2 },
    { 0, 3, 1 },
    { 4, 4, 6 },
};

// Réduire à un joueur à la fois ; les colonnes sont transposées pour réutiliser le LP.
double[,] Reduced(double[,] m, List<int> rows, List<int> cols, bool transpose = false)
{
    var a = new double[transpose ? cols.Count : rows.Count, transpose ? rows.Count : cols.Count];
    for (int i = 0; i < rows.Count; i++)
    for (int j = 0; j < cols.Count; j++)
        if (transpose)
            a[j, i] = m[rows[i], cols[j]];
        else
            a[i, j] = m[rows[i], cols[j]];
    return a;
}
(List<string> Trace, List<int> Rows, List<int> Cols) Idsds(
    double[,] a,
    double[,] b,
    string[] namesA,
    string[] namesB
)
{
    var rows = Enumerable.Range(0, a.GetLength(0)).ToList();
    var cols = Enumerable.Range(0, a.GetLength(1)).ToList();
    var trace = new List<string>();
    while (true)
    {
        bool removed = false;
        for (int player = 0; player < 2 && !removed; player++)
        {
            var living = player == 0 ? rows : cols;
            var names = player == 0 ? namesA : namesB;
            var m = player == 0 ? Reduced(a, rows, cols) : Reduced(b, rows, cols, true);
            for (int i = 0; i < living.Count; i++)
            {
                var (margin, mix) = DominanceMargin(m, i);
                if (margin > 1e-9)
                {
                    trace.Add(
                        $"Joueur {player + 1} supprime {names[living[i]]} : mixte [{string.Join(", ", mix.Select(x => x.ToString("F4")))}], marge {margin:F4}"
                    );
                    living.RemoveAt(i);
                    removed = true;
                    break;
                }
            }
        }
        if (!removed)
            break;
    }
    return (trace, rows, cols);
}
var namesG0A = new[] { "A", "B", "C" };
var namesG0B = new[] { "D", "E", "F" };
var g0 = Idsds(g0a, g0b, namesG0A, namesG0B);
foreach (var line in g0.Trace)
    Console.WriteLine(line);
Console.WriteLine(
    $"Stratégies rationalisables : {string.Join(",", g0.Rows.Select(i => namesG0A[i]))} × {string.Join(",", g0.Cols.Select(i => namesG0B[i]))}"
);


Joueur 1 supprime C : mixte [0.5000, 0.5000, 0.0000], marge 0.5000


Joueur 2 supprime F : mixte [0.5000, 0.5000, 0.0000], marge 0.5000


Joueur 1 supprime B : mixte [1.0000, 0.0000], marge 1.0000


Joueur 2 supprime E : mixte [1.0000, 0.0000], marge 3.0000


Stratégies rationalisables : A × D


### Contrôle de l’équilibre

Comparons le survivant de l’IDSDS aux équilibres énumérés par Nashpy.


In [5]:
// Nashpy énumère les supports ; les matrices et l'interprétation restent en C#.
List<(double[] Row, double[] Col)> NashEquilibria(double[,] a, double[,] b)
{
    using (Py.GIL())
    {
        using var np = Py.Import("numpy");
        using var nash = Py.Import("nashpy");
        using var builtins = Py.Import("builtins");
        using var ma = Matrix(a);
        using var mb = Matrix(b);
        using var na = np.InvokeMethod("array", new PyObject[] { ma });
        using var nb = np.InvokeMethod("array", new PyObject[] { mb });
        using var factory = nash.GetAttr("Game");
        using var game = factory.Invoke(new[] { na, nb });
        using var iter = game.InvokeMethod("support_enumeration");
        using var list = builtins.InvokeMethod("list", new[] { iter });
        var result = new List<(double[], double[])>();
        for (int i = 0; i < list.Length(); i++)
        {
            using var pair = list[i];
            using var r = pair[0];
            using var c = pair[1];
            var row = new double[(int)r.Length()];
            var col = new double[(int)c.Length()];
            for (int j = 0; j < row.Length; j++)
            {
                using var x = r[j];
                row[j] = x.As<double>();
            }
            for (int j = 0; j < col.Length; j++)
            {
                using var x = c[j];
                col[j] = x.As<double>();
            }
            result.Add((row, col));
        }
        return result;
    }
}
var g0eq = NashEquilibria(g0a, g0b);
foreach (var eq in g0eq)
    Console.WriteLine(
        $"Nash G0 : lignes [{string.Join(",", eq.Row.Select(x => x.ToString("F4")))}], colonnes [{string.Join(",", eq.Col.Select(x => x.ToString("F4")))}]"
    );
Console.WriteLine($"Nombre d'équilibres : {g0eq.Count}");


Nash G0 : lignes [1.0000,0.0000,0.0000], colonnes [1.0000,0.0000,0.0000]


Nombre d'équilibres : 1


### Interprétation : rationalisables uniques ⇒ équilibre unique

Les quatre suppressions recalculées épousent la trace du manuel — $C$ chassée par $\frac{1}{2}A + \frac{1}{2}B$ (marge $0{,}5$ sur chaque colonne), puis $F$ par $\frac{1}{2}D + \frac{1}{2}E$, puis $B$ par $A$ (dominance pure), puis $E$ par $D$ — et le survivant est le singleton $(A, D)$, exactement le seul équilibre que `nashpy` énumère. Dans ce jeu, la rationalité itérée **suffit** à prédire l'issue : l'hypothèse d'équilibre n'ajoute rien. La section suivante mesure à quel point ce cas est exceptionnel.

### Exercice 1 : IDSDS cardinal sur une 3×3

Complétez `exercice1` pour retourner les tableaux de noms survivants des deux joueurs du jeu 5.12 ; `Idsds` fournit la procédure.


In [6]:
var a512 = new double[,]
{
    { 3, 2, 2 },
    { 5, 1, 2 },
    { 9, 1, 3 },
};
var b512 = new double[,]
{
    { 5, 0, 2 },
    { 2, 2, 1 },
    { 0, 5, 2 },
};
var rows512 = new[] { "A", "B", "C" };
var cols512 = new[] { "L", "M", "R" };

// Exercice 1 : renvoyer les noms des stratégies survivantes des deux joueurs.
Func<(string[] P1, string[] P2)?> exercice1 = () =>
{
    // TODO : appliquer l'IDSDS cardinal au jeu 5.12.
    return null;
};
var expected1 = Idsds((double[,])a512.Clone(), (double[,])b512.Clone(), rows512, cols512);
var answer1 = exercice1();
if (answer1 == null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    var expected = expected1;
    if (
        answer1.Value.P1 == null
        || answer1.Value.P2 == null
        || !answer1.Value.P1.SequenceEqual(expected.Rows.Select(i => rows512[i]))
        || !answer1.Value.P2.SequenceEqual(expected.Cols.Select(i => cols512[i]))
    )
        throw new Exception("Survivants incorrects ou réponse partielle");
    Console.WriteLine("Exercice 1 : survivants vérifiés");
}


Exercice 1 à compléter


## 4. Le pont vers le palier 04 : ce que la rationalité ne délie pas

Les quatre jeux emblématiques du palier 04 — Dilemme du Prisonnier, Battle of the Sexes, chasse au cerf, Matching Pennies — passent au même banc : ensembles rationalisables (IDSDS cardinal) contre équilibres de Nash (mixtes compris). La question : l'inclusion est-elle stricte ?
Pour comparer des objets de même nature, les noms affichés désignent les actions pures survivantes ; on considère ensuite les distributions dont le support est inclus dans ces actions. Les profils de Nash, purs ou mixtes, sont comparés à ce produit de simplexes.


In [7]:
// Les quatre jeux conservent leurs matrices, et chaque verdict combine IDSDS et Nashpy.
var games = new[]
{
    (
        "Dilemme du Prisonnier",
        new double[,]
        {
            { 3, 0 },
            { 5, 1 },
        },
        new double[,]
        {
            { 3, 5 },
            { 0, 1 },
        },
        new[] { "Coopérer", "Défaire" },
        new[] { "Coopérer", "Défaire" }
    ),
    (
        "Battle of the Sexes",
        new double[,]
        {
            { 3, 0 },
            { 0, 2 },
        },
        new double[,]
        {
            { 2, 0 },
            { 0, 3 },
        },
        new[] { "Opéra", "Football" },
        new[] { "Opéra", "Football" }
    ),
    (
        "Chasse au cerf",
        new double[,]
        {
            { 4, 0 },
            { 3, 3 },
        },
        new double[,]
        {
            { 4, 3 },
            { 0, 3 },
        },
        new[] { "Cerf", "Lièvre" },
        new[] { "Cerf", "Lièvre" }
    ),
    (
        "Matching Pennies",
        new double[,]
        {
            { 1, -1 },
            { -1, 1 },
        },
        new double[,]
        {
            { -1, 1 },
            { 1, -1 },
        },
        new[] { "Pile", "Face" },
        new[] { "Pile", "Face" }
    ),
};
foreach (var (name, a, b, ra, cb) in games)
{
    var rat = Idsds(a, b, ra, cb);
    var eq = NashEquilibria(a, b);
    int pure = eq.Count(x => x.Row.Max() > 1 - 1e-9 && x.Col.Max() > 1 - 1e-9);
    Console.WriteLine(
        $"{name} : rationalisables [{string.Join(",", rat.Rows.Select(i => ra[i]))}] × [{string.Join(",", rat.Cols.Select(i => cb[i]))}] ; Nashpy {pure} pur(s), {eq.Count - pure} mixte(s) ; {(rat.Rows.Count == 2 && rat.Cols.Count == 2 ? "inclusion stricte" : "purge singleton")}"
    );
}


Dilemme du Prisonnier : rationalisables [Défaire] × [Défaire] ; Nashpy 1 pur(s), 0 mixte(s) ; purge singleton


Battle of the Sexes : rationalisables [Opéra,Football] × [Opéra,Football] ; Nashpy 2 pur(s), 1 mixte(s) ; inclusion stricte


Chasse au cerf : rationalisables [Cerf,Lièvre] × [Cerf,Lièvre] ; Nashpy 2 pur(s), 1 mixte(s) ; inclusion stricte


Matching Pennies : rationalisables [Pile,Face] × [Pile,Face] ; Nashpy 0 pur(s), 1 mixte(s) ; inclusion stricte


### Interprétation : trois inclusions strictes sur quatre jeux

Le tableau le dit en une colonne : seul le **Dilemme du Prisonnier** voit la rationalité itérée refermer l'équilibre (la dominance de Défaire purge les deux lignes/colonnes, et le survivant unique *est* l'équilibre) ; sur les trois autres jeux, **aucune** stratégie n'est dominée — même par une mixte — donc l'ensemble rationalisable reste le produit complet des stratégies, tandis que la colonne des équilibres compte trois points pour le BoS comme pour la chasse au cerf (deux purs et un mixte dans les deux cas), et un seul point, mixte, pour Matching Pennies. L'écart mesuré entre les deux colonnes est exactement ce que l'hypothèse d'équilibre **ajoute** à la rationalité commune : la cohérence mutuelle des conjectures, chacune étant *confirmée* par la stratégie de l'autre — la lecture « mixte = croyance » du carnet 04f.

### Exercice 2 : mixture de rationalisables

Complétez `exercice2` avec un jeu d’au moins 3×2 et une mixture sur deux lignes rationalisables qui n’est jamais meilleure réponse. Retournez un `ContreExemple` avec matrices, mixture et explication.


In [8]:
record ContreExemple(double[,] P1, double[,] P2, double[] Mixture, string Explication);
// Exercice 2 : deux lignes rationalisables, mixture jamais meilleure réponse.
Func<ContreExemple> exercice2 = () =>
{
    // TODO : construire un jeu d'au moins 3×2 et une mixture sur deux lignes.
    return null;
};
var answer2 = exercice2();
if (answer2 == null)
    Console.WriteLine("Exercice 2 à compléter");
else
{
    var a = answer2.P1;
    var b = answer2.P2;
    var mix = answer2.Mixture;
    if (
        a == null
        || b == null
        || mix == null
        || a.GetLength(0) < 3
        || a.GetLength(1) < 2
        || a.GetLength(0) != b.GetLength(0)
        || a.GetLength(1) != b.GetLength(1)
        || mix.Length != a.GetLength(0)
        || a.Cast<double>().Any(x => !double.IsFinite(x))
        || b.Cast<double>().Any(x => !double.IsFinite(x))
        || mix.Any(x => !double.IsFinite(x) || x < 0)
        || Math.Abs(mix.Sum() - 1) > 1e-9
        || mix.Count(x => x > 1e-9) != 2
        || string.IsNullOrWhiteSpace(answer2.Explication)
    )
        throw new Exception("Forme, mixture ou explication invalide");
    var rn = Enumerable.Range(0, a.GetLength(0)).Select(i => $"r{i}").ToArray();
    var cn = Enumerable.Range(0, a.GetLength(1)).Select(i => $"c{i}").ToArray();
    var rat = Idsds(a, b, rn, cn);
    if (Enumerable.Range(0, mix.Length).Any(i => mix[i] > 1e-9 && !rat.Rows.Contains(i)))
        throw new Exception("Support non rationalisable");
    var augmented = new double[a.GetLength(0) + 1, a.GetLength(1)];
    for (int i = 0; i < a.GetLength(0); i++)
    for (int j = 0; j < a.GetLength(1); j++)
    {
        augmented[i, j] = a[i, j];
        augmented[a.GetLength(0), j] += mix[i] * a[i, j];
    }
    if (BestResponseMargin(augmented, a.GetLength(0)).Margin >= -1e-9)
        throw new Exception("La mixture reste meilleure réponse");
    Console.WriteLine("Exercice 2 : contre-exemple vérifié");
}



Exercice 2 à compléter


## 5. La lecture épistémique : rationalité commune de croyance

La **remarque 5.5** du manuel donne la définition opératoire : un joueur est *rationnel* si sa stratégie pure est une meilleure réponse à sa croyance sur l'adversaire ; à deux joueurs, une croyance est une distribution — le même objet qu'une stratégie mixte. En itérant — « je crois que tu es rationnel, tu crois que je crois que tu es rationnel, … » — on obtient la **rationalité commune de croyance**, et le manuel annonce (ch. 9, développé au carnet 02d) le théorème : sous rationalité commune, seuls des profils rationalisables peuvent être joués.

La **remarque 5.6** pose la limite, et elle n'est pas décorative : l'itération exige que les préférences **cardinales** (vNM) des joueurs soient connaissance commune. Un adversaire qui ne connaîtrait que le *classement ordinal* des issues ne pourrait pas déduire que la stratégie $B$ de la Table 5.8 est irrationnelle — la mixte $\frac{1}{3}A + \frac{2}{3}C$ qui la condamne n'existe que dans les valeurs, pas dans l'ordre. Exiger la connaissance commune d'utilités cardinales est une hypothèse forte (le manuel lui-même la dit « souvent très irréaliste ») : la rationalisabilité hérite exactement de cette fragilité, et c'est un enseignement du carnet autant qu'un théorème.

### Exercice 3 : à trois joueurs, la croyance corrélée (Bonanno, exercice 5.14)

Le jeu à trois joueurs de l'exercice 5.14 du manuel, où seuls les paiements du Joueur 1 sont montrés — le Joueur 1 choisit la ligne (A, B, C, D), le Joueur 2 la colonne (E, F) et le Joueur 3 le panneau (G, H) :

| P1 \ P2 | E | F | | E | F |
|---|---|---|---|---|---|
| **A** | 3 | 0 | **A** | 0 | 0 |
| **B** | 0 | 3 | **B** | 3 | 0 |
| **C** | 0 | 0 | **C** | 0 | 3 |
| **D** | 2 | 0 | **D** | 0 | 2 |

(le panneau de gauche correspond à « Joueur 3 joue G », celui de droite à « Joueur 3 joue H »)

1. Vérifiez que si le Joueur 1 croit à ½ que l'événement « J2 joue E **et** J3 joue G » se réalisera, et à ½ que « J2 joue F **et** J3 joue H » se réalisera, alors **D** est sa meilleure réponse.
2. Montrez qu'il n'existe **aucun** profil mixte **indépendant** $(p, 1{-}p)$ sur (E, F) et $(q, 1{-}q)$ sur (G, H) des Joueurs 2 et 3 contre lequel D soit une meilleure réponse de P1 — c'est la démonstration que le théorème de Pearce ne s'étend pas tel quel au-delà de deux joueurs : une croyance peut être **corrélée**, un profil mixte ne peut pas.
Retournez une croyance 2×2 valide, la marge maximale indépendante **exacte** et une explication ; un simple balayage donne une approximation à justifier.

Complétez `exercice3` avec une croyance jointe 2×2 et la marge maximale indépendante exacte, ainsi qu’une explication.


In [9]:
var panelG = new double[,]
{
    { 3, 0 },
    { 0, 3 },
    { 0, 0 },
    { 2, 0 },
};
var panelH = new double[,]
{
    { 0, 0 },
    { 3, 0 },
    { 0, 3 },
    { 0, 2 },
};

record Croyance(double[,] Jointe, double MargeIndependante, string Explication);
// Exercice 3 : une croyance corrélée et la marge optimale sous indépendance.
Func<Croyance> exercice3 = () =>
{
    // TODO : construire une croyance et justifier la borne exacte.
    return null;
};
var answer3 = exercice3();
if (answer3 == null)
    Console.WriteLine("Exercice 3 à compléter");
else
{
    var belief = answer3.Jointe;
    if (
        belief == null
        || belief.GetLength(0) != 2
        || belief.GetLength(1) != 2
        || belief.Cast<double>().Any(x => !double.IsFinite(x) || x < 0)
        || Math.Abs(belief.Cast<double>().Sum() - 1) > 1e-9
        || !double.IsFinite(answer3.MargeIndependante)
        || string.IsNullOrWhiteSpace(answer3.Explication)
    )
        throw new Exception("Croyance ou réponse invalide");
    var gardeG = new double[,]
    {
        { 3, 0 },
        { 0, 3 },
        { 0, 0 },
        { 2, 0 },
    };
    var gardeH = new double[,]
    {
        { 0, 0 },
        { 3, 0 },
        { 0, 3 },
        { 0, 2 },
    };
    var gains = Enumerable
        .Range(0, 4)
        .Select(i =>
            Enumerable
                .Range(0, 2)
                .Sum(j => gardeG[i, j] * belief[j, 0] + gardeH[i, j] * belief[j, 1])
        )
        .ToArray();
    // Grille de contrôle indépendante de la réponse fournie.
    double borneGrille = double.NegativeInfinity;
    for (int pi = 0; pi <= 300; pi++)
        for (int qi = 0; qi <= 300; qi++)
        {
            double p = pi / 300.0, q = qi / 300.0;
            var produit = new double[,] { { p*q, p*(1-q) }, { (1-p)*q, (1-p)*(1-q) } };
            var valeurs = Enumerable.Range(0,4).Select(k => Enumerable.Range(0,2).Sum(j => gardeG[k,j]*produit[j,0] + gardeH[k,j]*produit[j,1])).ToArray();
            borneGrille = Math.Max(borneGrille, valeurs[3] - valeurs.Take(3).Max());
        }
    if (
        gains[3] < gains.Take(3).Max() - 1e-9
        || Math.Abs(answer3.MargeIndependante - borneGrille) > 1e-9
    )
        throw new Exception("Meilleure réponse ou marge incorrecte");
    Console.WriteLine("Exercice 3 : croyance et marge vérifiées");
}



Exercice 3 à compléter


## À retenir

| Concept | Énoncé | Où il est mesuré |
|---|---|---|
| Dominance par une mixte | $\frac{1}{3}A + \frac{2}{3}C$ rapporte $\frac{4}{3}$ contre les deux colonnes, B en rapporte 1 : B condamnée sans dominance pure | Section 1, Table 5.8 |
| Théorème de Pearce (1984) | jamais-meilleure-réponse $\iff$ dominée par une mixte (2 joueurs) | Section 2 : deux LP indépendants, 3/3 accords, mixte $(\frac{1}{3}, 0, \frac{2}{3})$ retrouvée par le solveur |
| IDSDS cardinal | itérer la suppression des dominées-par-mixte ; survivants = rationalisables | Section 3 : Figure 5.9 rejouée, 4 suppressions, survivant $(A, D)$ |
| Rationalisables $\supseteq$ équilibres | stricte sur les trois jeux étudiés où aucune dominance n'amorce la purge | Section 4 : PD égalité ; BoS, cerf, pennies inclusion stricte |
| Condition épistémique | rationalité commune de croyance $\Rightarrow$ profils rationalisables ; exige les vNM en connaissance commune | Section 5 (renvois 02d, 04f) |

**Quatre mesures à retenir.**

- **B est condamnée sans aucune dominance pure.** Ni `A` ni `C` ne domine strictement B (`A − B = [-1., 3.]`, `C − B = [1., −1.]`), mais la mixte `1/3 A + 2/3 C` rapporte `1.333333` contre `1` dans les deux colonnes : la marge minimale sur B vaut `0.333333`, et le LP retrouve la mixte dominatrice `[0.333333 0. 0.666667]`.
- **Le théorème de Pearce est vérifié dans les deux sens, avec 3/3 accords.** Pour `A`, la marge de meilleure réponse vaut `3.000000` et la marge de dominance mixte `-0.000000` (meilleure réponse possible, non dominée) ; pour `B`, `-0.333333` et `0.333333` (jamais MR **et** dominée) ; pour `C`, `1.000000` et `-0.000000`.
- **L'IDSDS écarte quatre actions et laisse un seul survivant.** C (marge `0.5000`), puis F (`0.5000`), puis B (`1.0000`), puis E (`3.0000`) : les rationalisables finissent en `['A'] x ['D']`.
- **Rationalisables ⊇ équilibres, et l'inclusion est stricte sur les trois jeux étudiés où aucune dominance n'amorce la purge.** Battle of the Sexes, Chasse au cerf et Matching Pennies sont tous en « inclusion stricte » ; seul le Dilemme du Prisonnier se réduit à un singleton qui est l'équilibre.


## Vérifiez votre compréhension

1. Aucune stratégie pure ne domine strictement B, et pourtant B est éliminée. Par quel mécanisme, et avec quelle preuve chiffrée ?

   <details><summary>Réponse</summary>

   Par la dominance d'une **mixte** : la mixte `1/3 A + 2/3 C` rapporte `1.333333` contre `1` dans les deux colonnes, soit une marge minimale sur B de `0.333333` (> 0). Le LP retrouve cette dominante sous la forme `[0.333333 0. 0.666667]`, c'est-à-dire exactement `1/3, 0, 2/3` — la vérification est numérique et non déclarative.

   </details>

2. Pour la ligne `A`, le tableau donne marge de meilleure réponse `3.000000` et marge de dominance mixte `-0.000000`. Que dit la valeur `-0.000000` ?

   <details><summary>Réponse</summary>

   Qu'aucune mixte ne domine A : la marge vaut zéro (au bruit numérique près, d'où le `-0.000000` signé). A est donc une meilleure réponse pour certaines croyances, et n'est pas éliminée par l'IDSDS. La même lecture vaut pour `C` (`1.000000` et `-0.000000`). La ligne `B` est la seule où les deux marges sont non nulles — et c'est la seule éliminée.

   </details>

3. L'IDSDS élimine C, F, B, E et laisse `['A'] x ['D']`. Sur le jeu `G0`, l'ensemble des équilibres de Nash vaut lui aussi `[(A, D)]`. Que prouve cette coïncidence — et que ne prouve-t-elle pas ?

   <details><summary>Réponse</summary>

   Elle prouve que **sur ce jeu**, rationalisables et équilibres coïncident : la purge effectue quatre suppressions et le verdict du carnet le note « purge → singleton = équilibre ». Elle ne prouve pas l'égalité en général : sur les trois autres jeux du palier, le verdict est « inclusion stricte » — les rationalisables contiennent les équilibres et les débordent. Une coïncidence mesurée sur un jeu ne vaut pas théorème.

   </details>

4. La rationalisabilité exige que les préférences **cardinales** des joueurs soient connaissance commune ; le manuel lui-même parle d'une hypothèse « souvent très irréaliste ». Le carnet peut-il soutenir que la rationalisabilité est un ensemble « d'équilibres dégradés » ?

   <details><summary>Réponse</summary>

   Non, et la remarque 5.6 le dit : exiger la connaissance commune d'utilités cardinales est une hypothèse forte, mais ce n'est pas la même chose que supposer la rationalité elle-même. La rationalisabilité relâche la **cohérence mutuelle** (une stratégie est meilleure réponse à une croyance, pas à l'action observée) tout en gardant les préférences cardinales en hypothèse — d'où une fragilité **différente** de celle de Nash, que le carnet assume comme un enseignement au même titre qu'un théorème.

   </details>


## Pour aller plus loin

**Carnet suivant.** [GameTheory-04k — Équilibre corrélé](GameTheory-04k-Equilibre-Correle-Python.ipynb) -- le prolongement du palier : quand la cohérence mutuelle ne suffit plus, on ajoute la **recommandation commune**, et l'on mesure ce que la corrélation crée.

**Lectures.**

- Les références complètes du carnet sont dans la section « Références académiques » : Pearce (1984) pour le théorème 5.3, Bonanno (2015) ch. 5.4 (Table 5.8, théorème 5.3, définition 5.6, Figure 5.9, remarques 5.5-5.6, exercices 5.12-5.14) et von Neumann & Morgenstern (1944) pour les paiements cardinaux que la section 1 suppose — construits au carnet [GameTheory-04i](GameTheory-04i-Utilite-Esperee-Risque-Python.ipynb).
- Le chapitre 9 du manuel, develops au carnet **02d**, porte le théorème annoncé en section 5 : sous rationalité commune de croyance, seuls des profils rationalisables peuvent être joués. C'est le prolongement naturel de ce carnet.

**Carnets voisins.**

- [GameTheory-04 — Nash](GameTheory-04-NashEquilibrium-Python.ipynb) -- le track principal du palier : l'équilibre comme cohérence mutuelle, dont ce carnet montre ce qu'il reste quand on n'en garde que la rationalité.
- [GameTheory-04i — Utilité espérée et aversion au risque](GameTheory-04i-Utilite-Esperee-Risque-Python.ipynb) -- d'où viennent les utilités cardinales que la dominance par mixte suppose.
- [GameTheory-04k — Équilibre corrélé](GameTheory-04k-Equilibre-Correle-Python.ipynb) -- le relais suivant : la recommandation commune en place de la croyance.


## Références académiques

Bibliographie héritée du [carnet Python](GameTheory-04j-Rationalisabilite-Python.ipynb) : Pearce (1984), Bonanno (2015), von Neumann et Morgenstern (1944), sans nouvelle lecture revendiquée.


***